In [ ]:
# Set up for running selenium in Google Colab
%%shell
sudo apt -y update
sudo apt install -y wget curl unzip
wget http://archive.ubuntu.com/ubuntu/pool/main/libu/libu2f-host/libu2f-udev_1.1.4-1_all.deb
dpkg -i libu2f-udev_1.1.4-1_all.deb
wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
dpkg -i google-chrome-stable_current_amd64.deb
CHROME_DRIVER_VERSION=`curl -sS chromedriver.storage.googleapis.com/LATEST_RELEASE`
wget -N https://chromedriver.storage.googleapis.com/$CHROME_DRIVER_VERSION/chromedriver_linux64.zip -P /tmp/
unzip -o /tmp/chromedriver_linux64.zip -d /tmp/
chmod +x /tmp/chromedriver
mv /tmp/chromedriver /usr/local/bin/chromedriver
pip install selenium
pip install chromedriver-autoinstaller

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [731 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [108 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [734 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 1,802 kB in 2s (803 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
8 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from selenium import webdriver
import chromedriver_autoinstaller  # setup chrome options
import sys
import re
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

In [ ]:
# Function to initialize the WebDriver
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless') # ensure GUI is off
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')  # set path to chromedriver as per your configuration
    chromedriver_autoinstaller.install()  # set the target URL
    driver = webdriver.Chrome(options=chrome_options)

    return driver

In [ ]:
# Function to get user profile links and additional information from the user's profile page
def get_user_location(profile_link):
    driver = initialize_driver()
    driver.get(profile_link)

    # Wait for the user's profile page to load
    time.sleep(3)

    # Extract additional information from the user's profile page
    location = driver.find_element(By.CLASS_NAME, 'profile__user-location').text.split(',')[-1].lstrip()

    driver.quit()

    return location

In [ ]:
# Function to scrape the Kaggle Rankings table
def scrape_kaggle_rankings(url):

    users_df = pd.DataFrame(columns=['Username','Display_Name','Image_Path'])

    driver = initialize_driver()
    driver.get(url)

    # Get the div element to scroll
    data_grid_div = driver.find_element(By.CLASS_NAME, 'MuiDataGrid-virtualScroller')

    # Get the initial table height
    prev_height = driver.execute_script("return arguments[0].scrollHeight;", data_grid_div)

    # Set the time limit for scrolling (30 minutes)
    timeout = time.time() + 30 * 60

    while time.time() < timeout:

        # Extract data from the table rows
        rows = driver.find_elements(By.CLASS_NAME, 'MuiDataGrid-row')
        for row in rows:
            # Extract data from each cell (adjust the column index based on your table structure)
            cells = row.find_elements(By.CLASS_NAME, 'MuiDataGrid-cell')
            if len(cells) > 0:
                rank = cells[0].text
                display_name = cells[1].text
                image_path = re.search('\(([^)]+)', cells[1].find_element(By.CLASS_NAME, 'fHXWVA').get_attribute('style')).group(1)
                profile_link = cells[1].find_element(By.TAG_NAME, 'a').get_attribute('href')
                username = profile_link[profile_link.rindex('/')+1:]

                users_df.loc[len(users_df)] = [username, display_name, image_path]

                print(f"Username: {username}, Display_Name: {display_name}, Image_Path: {image_path}")

        # Scroll to the bottom of the div
        driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", data_grid_div)

        # Wait for a short time to allow new content to load (adjust sleep time if needed)
        time.sleep(1)

        # Check if new content is loaded by comparing the current height with the previous height
        current_height = driver.execute_script("return arguments[0].scrollHeight;", data_grid_div)

        if current_height == prev_height:
            break  # Exit the loop if no new content is loaded
        else:
            prev_height = current_height
    driver.quit()

    return users_df.drop_duplicates(subset=['Username'])

In [ ]:
ranking_groups=['competitions','datasets','notebooks','discussion']

users_df_merged = pd.DataFrame(columns=['Username','Display_Name','Image_Path'])

for group in ranking_groups:
  users_df_merged = pd.concat([users_df_merged, scrape_kaggle_rankings(f'https://www.kaggle.com/rankings?group={group}')])

users_df_merged.drop_duplicates(subset=['Username'],inplace=True)

Streaming output truncated to the last 5000 lines.
Username: masakick, Display_Name: masakick07, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/default-thumb.png"
Username: romanvan8, Display_Name: Riley Zhu, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/7315931-kg.jpg"
Username: wikunia, Display_Name: Ole Kröger, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/300833-gr.jpg"
Username: mgchbot, Display_Name: mgchbot, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/399236-kg.jpeg"
Username: gody7334, Display_Name: gody7334, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/default-thumb.png"
Username: huangzihan, Display_Name: ZH, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/1849139-kg.jpg"
Username: jongoslst, Display_Name: JongOS, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/default-thumb.png"
Username: arthurds, Display_Name:

Username: thedevastator, Display_Name: The Devastator, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/10654180-kg.png"
Username: andrewmvd, Display_Name: Larxel, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/793761-kg.jpg"
Username: iamsouravbanerjee, Display_Name: Sourav Banerjee, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/7024923-kg.png"
Username: mpwolke, Display_Name: Marília Prata, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/3012786-kg.jpg"
Username: surajjha101, Display_Name: SJ, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/9590174-kg.jpg"
Username: ruchi798, Display_Name: Ruchi Bhatia, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/3309826-kg.jpeg"
Username: shivamb, Display_Name: Shivam Bansal, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/1571785-kg.jpeg"
Username: timoboz, Display_Name: Timo Bozsolik, Image

Streaming output truncated to the last 5000 lines.
Username: mostafahabibi1994, Display_Name: Mostafa Habibi, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/13807665-kg.JPG"
Username: gokulprasantht, Display_Name: Gokulprasanth T, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/13052499-kg.jpg"
Username: bcruise, Display_Name: Bill Cruise, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/195475-gr.jpg"
Username: fangya, Display_Name: Fangya, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/1185728-kg.JPG"
Username: sytuannguyen, Display_Name: Sy-Tuan Nguyen, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/4199239-kg.png"
Username: nxrprime, Display_Name: Trigram, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/4321234-kg.jpg"
Username: harshsingh2209, Display_Name: Harsh Singh, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/8064850-kg

Username: cdeotte, Display_Name: Chris Deotte, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/1723677-kg.jpg"
Username: mpwolke, Display_Name: Marília Prata, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/3012786-kg.jpg"
Username: ravi20076, Display_Name: Ravi Ramakrishnan, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/8273630-kg.jpg"
Username: thedevastator, Display_Name: The Devastator, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/10654180-kg.png"
Username: hengck23, Display_Name: , Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/113660-fb.jpg"
Username: cpmpml, Display_Name: CPMP, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/75976-gr.jpg"
Username: ambrosm, Display_Name: AmbrosM, Image_Path: "https://storage.googleapis.com/kaggle-avatars/thumbnails/default-thumb.png"
Username: carlmcbrideellis, Display_Name: Carl McBride Ellis, Image_Path: "ht

In [ ]:
users_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 14620 entries, 0 to 3297
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Username      14620 non-null  object
 1   Display_Name  14620 non-null  object
 2   Image_Path    14620 non-null  object
dtypes: object(3)
memory usage: 456.9+ KB


In [ ]:
users_df_merged.head()

,Username,Display_Name,Image_Path
0,bestfitting,bestfitting,"""https://storage.googleapis.com/kaggle-avatars..."
1,philippsinger,Psi,"""https://storage.googleapis.com/kaggle-avatars..."
2,christofhenkel,Dieter,"""https://storage.googleapis.com/kaggle-avatars..."
3,cdeotte,Chris Deotte,"""https://storage.googleapis.com/kaggle-avatars..."
4,haqishen,Qishen Ha,"""https://storage.googleapis.com/kaggle-avatars..."


In [ ]:
users_df_merged.to_csv(r'kaggle_users.csv')